In [10]:
!pip install --upgrade google-meridian[colab,and-cuda,schema]

import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
from itertools import permutations
from google.colab import drive
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp
import sys
import os

# Mount a storage
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

!git clone --branch meridian_modeling https://github.com/pstat197/BlueAlpha3-Synergy-Analysis

df = pd.read_csv("/content/BlueAlpha3-Synergy-Analysis/data/monthly_mocha.csv")
df = df.loc[:, (df != 0).any()]

import sys
sys.path.append("/content/BlueAlpha3-Synergy-Analysis/scripts")
from geometric_mean import create_geometric_mean_interactions

df_mmm = create_geometric_mean_interactions(df)

print(df_mmm.head())

Mounted at /content/drive
fatal: destination path 'BlueAlpha3-Synergy-Analysis' already exists and is not an empty directory.
      date  subscriptions   meta_spend  meta_impressions  google_spend  \
0   8/4/25          15540  91538.06648          16572258   116667.9945   
1  7/28/25          14525  93840.18612          25300600   180486.9558   
2  7/21/25          16880  48403.06780          14099214   200817.3250   
3  7/14/25          20113  49470.96783          13652072   215770.9242   
4   7/7/25          16492  48948.28744          10121002   209231.9668   

   google_impressions  snapchat_spend  snapchat_impressions  tiktok_spend  \
0             6473132     94750.04035               3420454           0.0   
1             9487127     99447.23218               3235285           0.0   
2             7909118     84738.57435               4766750           0.0   
3             7789279     83204.40500               4022680           0.0   
4             6806878     82642.37271       

In [13]:
# Separate spend and impressions columns

spend_cols = [c for c in df_mmm.columns if c.endswith("_spend")]
impressions_cols = [c for c in df_mmm.columns if c.endswith("_impressions")]

channels = sorted({
    c.replace("_spend", "").replace("_impressions", "")
    for c in (spend_cols + impressions_cols)
})

assert len(channels) == len(spend_cols) == len(impressions_cols)

# Add nuisance cols
df_mmm["time"] = pd.date_range(
    start="2020-01-01",
    periods=len(df_mmm),
    freq="D"
).astype(str)

# Required placeholders
df_mmm["revenue_per_conversion"] = 1.0
df_mmm["population"] = 1.0

# Create builder
kpi_col = "subscriptions"

builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column=kpi_col,
    default_revenue_per_kpi_column="revenue_per_conversion",
)

builder = (
    builder.with_kpi(df_mmm)
    .with_revenue_per_kpi(df_mmm)
    .with_population(df_mmm)
    .with_media(
        df_mmm,
        media_cols=impressions_cols,
        media_spend_cols=spend_cols,
        media_channels=channels
    )
)

mmm_data = builder.build()

# Initialize model
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=0.2,
        scale=0.9,
        name=constants.ROI_M
    )
)

model_spec = spec.ModelSpec(
    prior=prior,
    enable_aks=True
)

mmm = model.Meridian(
    input_data=mmm_data,
    model_spec=model_spec
)

# Sample from the model
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

/usr/local/lib/python3.12/dist-packages/meridian/data/input_data_builder.py:715: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:74: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution

In [14]:
# Model diagnostics
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

/tmp/ipykernel_5276/1500263722.py:2: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Metric check,Status,Recommended action
Convergence,Pass,"The model has likely converged, as all parameters have R-hat values < 1.2."
Baseline,Review,"The posterior probability that the baseline is negative is 0.37. This indicates that the baseline time series occasionally dips into negative values. We recommend visually inspecting the baseline time series in the Model Fit charts, but don't be overly concerned. An occasional, small dip may indicate minor statistical error, which is inherent in any model."
Bayesian p-value,Pass,The Bayesian posterior predictive p-value is 0.88. The observed total outcome is consistent with the model's posterior predictive distribution.
Goodness of fit,Pass,"R-squared = 0.8830, MAPE = 0.0568, and wMAPE = 0.0542. These goodness-of-fit metrics are intended for guidance and relative comparison."
Prior-posterior shift,Pass 10/10 channels passed,The model has successfully learned from the data. This is a positive sign that your data was informative.


In [15]:
# Two-page summary
mmm_summarizer = summarizer.Summarizer(mmm)

filepath = meridian_root
start_date = '2020-01-01'
end_date = '2020-03-01'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:3356: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(


Dataset,R-squared,MAPE,wMAPE
All Data,0.89,5%,5%
